### Two Pointer method 

1. What “two pointers” actually means

A pointer here usually means an index or a reference identifying a position.

The technique maintains two positions and moves them according to a rule, avoiding unnecessary comparisons.

For example, finding two numbers whose sum equals a target:
- Brute force: compare every pair → O(n^2).
- Two pointers on a sorted array: eliminate candidates systematically → O(n).

The important question is:
**What property allows me to move this pointer without losing a possible answer?**

That property might be sorted order, a valid window, an already-processed prefix, or a relative distance between two pointers.

Two pointers does not necessarily mean:
    - Exactly two variables.
    - Both pointers move on every iteration.
    - Both pointers move at the same speed.
    - The input must always be sorted.
    - The algorithm must always be O(n).

| Variant | Positions and movement | Typical purpose | Examples |
|---|---|---|---|
| Opposite ends | Start at opposite ends; move inward | Eliminate pairs or compare symmetry | Sorted Two Sum, palindrome |
| Two sequences | One pointer per input; advance selectively | Coordinate ordered sequences | Merge, intersection, interval intersection |
| Read/write | Both move forward; one scans, one writes | Modify an array in place | Remove duplicates, move zeroes |
| Sliding window | Left/right bound a contiguous region | Maintain a valid substring/subarray | Longest unique substring |
| Fast/slow | Move through a linked structure at different speeds | Detect cycles or locate the middle | Linked-list cycle, middle node |
| Fixed gap | One pointer stays a fixed distance ahead | Locate a position relative to the end | Remove nth node from end |
| Expand from centre | Start together or adjacent; move outward | Explore symmetry around a centre | Palindromic substrings |
| Partition boundaries | Track regions satisfying different conditions | Rearrange elements into groups | Partition around a pivot |
| Fixed anchor + pair | Fix one item; scan the remaining range | Reduce a higher-order search | 3Sum, 4Sum |

2. Opposite ends: eliminate impossible candidates

Worked example A: Two Sum in a sorted array

Problem: Given an ascending sorted array, return the zero-based indices of two distinct elements whose sum equals target. Return None if there is no solution.

In [ ]:
nums = [1, 2, 4, 6, 10]
target = 8

# Output: (1, 3), because nums[1] + nums[3] = 2 + 6 = 8 

# Try: 
nums = [1, 2, 3, 4, 6, 10]
target = 8

In [8]:
# naive / Brute force method 
def two_sum_naive(arr, target):
    n = len(arr)
    for i in range(n):
        for j in range(i+1, n):
            if arr[i] + arr[j] == target:
                return (i, j)
    return -1
            
arr = [1, 2, 3, 4, 6, 10]
two_sum_naive(arr, 8)

(1, 4)

For the above, Complexity
Time: O(n^2)
Space: O(1)

Now, Start with the smallest and largest remaining values 

In [ ]:
def two_sum_sorted(nums, target):
    left = 0 
    right = len(nums) - 1 

    while left < right:
        total = nums[left] + nums[right]

        if total == target:
            return left, right
        if total < target:
            left += 1 
        else: 
            right -= 1 
    return None

# Usage 
nums = [1, 2, 4, 6, 10]
# nums = [11, 1, 7, 2, 13, 4, 6, 10]
target = 8 
# nums = sorted(nums)
# print (nums)
print (two_sum_sorted(nums, target))

(1, 3)


Why is moving left safe when the sum is too small?
Suppose: nums[left] + nums[right] < target

Because nums[right] is the largest remaining value, pairing nums[left] with anything else in the remaining range produces an equal or smaller sum.

Therefore, no remaining pair using this left can work. Discard it.

The argument is symmetric when the sum is too large.

| Step | `left` | `right` | Values | Sum | Action |
|---|---:|---:|---|---:|---|
| 1 | 0 | 4 | 1, 10 | 11 | Decrease `right` |
| 2 | 0 | 3 | 1, 6 | 7 | Increase `left` |
| 3 | 1 | 3 | 2, 6 | 8 | Return `(1, 3)` |

Complexity: 
Time O(n) - because the array is traversed exactly once

Space O(1) - You just need a single variable to store the sum of values and compare it with the target

What happens on an unsorted array?

On an unsorted array, this elimination arguement fails. Sorting may help, but costs O(n logn) andchanges indices. For an unsorted Two Sum requiring original indices, a hash map is often a better choice. 


In [22]:
# An alternative to two pointer method - works on unsorted arrays too
# Using Hashmap (a dictionary) : O(n) Time and O(n) space 
# Just use a set() if you want to simply check whether sum of any of the two elements 
# added up to the target

def TwoSum(arr, target):
    hashMap = {}
    for i in range(len(arr)):
        complement = target - arr[i]

        if complement in hashMap:
            return [hashMap[complement], i]

        hashMap[arr[i]] = i
    
    return None

if __name__ == "__main__":
    arr = [0, -1, 2, -3, 1]
    target = -2
    print (TwoSum(arr, target))


[3, 4]


Worked example B: Container with most water

This example shows that the opposite-end pointers do not always require sorted input. 

Problem: You are given an integer array height of length n. THere are n vertical lines drawn such that the two enf-points of the i-th line are (i, 0) and (i, height[i])

Find the two lines that together with the x-axis form a container, such that the container contains the most water. You need to return the maximum volume of water a container can store.

The Intuition
The are of water trapped between any two vertical lines at indices left and right is calculated using the formula for the area of the rectangle:

    width = right - left 

    height = min(height[left], height[right])

    Area = width * height 

To maximize this area, you need to balance a wide distance between the bars with the tallest possible bar heights. 

i.e. choose the two elements of the array with the highest values and the two that are maxu=imum far apart on the arry (that is lowest in index value and another with highest value).


Optimal Solution using the Two Pointer method

If you check every single pair using nested loops, it takes O(N^2) time, which will time out. Instead, you can solve this in O(N) time using two pointers:

    1. Place one pointer at the very beginning (left = 0) and one at the very end (right = len(height) - 1). This gives you the maximum possible width right away.
    
    2. Calculate the area and update your maximum recorded area.
    
    3. The Core Move: To get a larger area, you must find a taller boundary. Since the amount of water is limited by the shorter of the two walls, keeping the shorter wall guarantees your area will only shrink as the width decreases. Therefore, you always move the pointer pointing to the shorter wall inward.



In [ ]:
def maxArea(heights):
    left = 0
    right = len(heights) - 1
    max_water = 0 

    while left < right:
        # calculate the current width 
        width = right - left 

        # calculate the current height (limited by the shorter wall)
        current_height = min(heights[left], heights[right])

        # calculate the area and update the global maximum 
        current_water = width * current_height
        max_water = max(max_water, current_water)

        # move the pointer of the shorter wall inward 
        if heights[left] < heights[right]:
            left += 1 
        else: 
            right -= 1
    return max_water

# --- Verification ---
heights = [1, 8, 6, 2, 5, 4, 8, 3, 7]
print(maxArea(heights))  
# Output: 49 
# (Formed between index 1 (height 8) and index 8 (height 7). Width = 7, Height = 7 -> 7 * 7 = 49)


49



Complexity Analysis

• Time Complexity: O(N) because the left and right pointers step toward each other, ensuring we look at each index at most once.
• Space Complexity: O(1) absolute constant space since we only store a few integer pointers and tracking variables.


3. Two inputs: merge, match, or advance past an impossible candidate

Think of two people reading two sorted lists. Each points to the next unread item. Compare those items to decide what to process next.

Worked example C: Merge two sorted arrays

Input: 
a = [1, 4, 7]
b = [2, 3, 8, 10]

Output: [1, 2, 3, 4, 7, 8, 10]

Note: Best answer in 88.E. Merge Sorted Array.ipynb

In [23]:
def merge_sorted(a, b):
    i = 0 
    j = 0 
    result = []

    # Both inputs still have unread elements
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1 

    # If b finished first, append everything remaining in a 
    while i < len(a):
        result.append(a[i])
        i += 1

    # If a finished first, appending everything remaining in b
    while j < len(b):
        result.append(b[j])
        j += 1 

    return result

nums1 = [1,2,3,0,0,0]
m = 3
nums2 = [2,5,6]
n = 3


print (f"First array: {nums1}")
print (f"Second array: {nums2}")
result = merge_sorted(nums1, nums2)
print (f"First array after merge: {result}")


First array: [1, 2, 3, 0, 0, 0]
Second array: [2, 5, 6]
First array after merge: [1, 2, 2, 3, 0, 0, 0, 5, 6]


Worked example D: Intersection of two sorted strings 

Return elements appearing in both arrays, preserving their shared multiplicity.


In [29]:
# This is not the optimal solution

a = [1, 2, 2, 4, 7]
b = [2, 2, 3, 7, 7]

# Output: [2, 2, 7]

def sorted_intersection(a, b):
    i = 0 
    j = 0 
    result = []

    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            result.append(a[i])
            i += 1
            j += 1
        elif a[i] < b[j]:
            i += 1
        else: 
            j += 1 
    return result

# Usage
nums1 = [4,9,5]
nums2 = [9,4,9,8,4]

print (sorted_intersection(nums1, nums2))
print (sorted_intersection(a, b))
        


[9]
[2, 2, 7]


### Use a hashset for optimal solution of this problem

In [38]:
def sorted_intersection_optimal(arr1, arr2):
    set2 = set(arr2)
    # result = set() 
    result = []

    for num in arr1:
        if num in set2:
            # result.add(num)
            result.append(num)
    return list(result)

# Usage

nums1 = [4,9,5]
nums2 = [9,4,9,8,4]
result = sorted_intersection_optimal(nums1, nums2)
print (result)
print (sorted_intersection_optimal(a, b))

[4, 9]
[2, 2, 7]


Worked example E: Remove duplicates from a sorted array

Problem: Modify the array in place so that each distinct value appears once in the first k positions. Return k.

In [ ]:
# Optimal solution 
def remove_duplicates(nums):
    write = 0 

    for read in range(len(nums)):
        if write == 0 or nums[read] != nums[write - 1]:
            nums[write] = nums[read]
            write += 1 

    return write 


[9, 4]


Invariant: nums[:write] contains the correct unique output for the input processed so far.

Why does overwriting not destroy unread data?

write never gets ahead of read, so writes only affect the current or previously processed positions.

Complexity: O(n) time, O(1) extra space.

This pattern also handles:
- Remove a specified value.
- Keep elements satisfying a predicate.
- Move zeroes.
- Allow at most two copies of each sorted value.
- In-place string compression.
A distinction worth remembering: This is sometimes called “slow/fast pointers,” but its reasoning differs from linked-list cycle detection. Here, one pointer scans and the other builds output.

5. Sliding Window: Two pointers enclosing a contiguous region 

A window represents a substring or subarray:
    nums[left:right + 1]

Typically:

    1. Expand by moving right.

    2. Update window state.

    3. Move left while a condition requires shrinking.

    4. Record an answer at the appropriate point.

There are two common optimisation shapes:

| Goal | Typical action |
|---|---|
| Longest window satisfying a restriction | Expand; shrink until valid; record length |
| Shortest window meeting a requirement | Expand until valid; record and shrink while valid |

Worked example F: Longest substring withot repeated characters 
